# Lab 4 lab solutions

## SQLAlchemy and vector search

In [1]:
from sqlalchemy.engine import URL

db_url = URL.create(
    drivername="postgresql+psycopg",
    username="postgres",
    password="password",
    host="localhost",
    port=5555,
    database="similarity_search_service_db"
)


In [2]:
from typing import List

from pgvector.sqlalchemy import Vector
from sqlalchemy import Integer, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column


# Create the base class for the table definition
class Base(DeclarativeBase):
    __abstract__ = True


# Create the table definition
class Images(Base):
    __tablename__ = "images"
    VECTOR_LENGTH = 512

    # primary key
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    # image path - we will use it to store the path to the image file, after similarity search we can use it to retrieve the image and display it
    image_path: Mapped[str] = mapped_column(String(256))
    # image embedding - we will store the image embedding in this column, the image embedding is a list of 512 floats this is the output of the sentence transformer model
    image_embedding: Mapped[List[float]] = mapped_column(Vector(VECTOR_LENGTH))


In [6]:
from sqlalchemy import create_engine

engine = create_engine(db_url)

Base.metadata.create_all(engine)



In [9]:
import numpy as np
from sqlalchemy import Engine, select
from sqlalchemy.orm import Session


# reusable function to insert data into the table
def insert_image(engine: Engine, image_path: str, image_embedding: list[float]):
    with Session(engine) as session:
        # create the image object
        image = Images(image_path=image_path, image_embedding=image_embedding)
        # add the image object to the session
        session.add(image)
        # commit the transaction
        session.commit()


# insert some data into the table
N = 100
for i in range(N):
    image_path = f"image_{i}.jpg"
    image_embedding = np.random.rand(512).tolist()
    insert_image(engine, image_path, image_embedding)

# select first image from the table
with Session(engine) as session:
    image = session.query(Images).first()


# Note: pgvector exposes *distances*, not similarities (lower = more similar).
# To rank, sort ascending by `cosine_distance`. To filter by a similarity threshold,
# use `1 - cosine_distance > threshold`.


# order the K images by ascending cosine distance to the first image (smallest distance = most similar)
def find_k_images(engine: Engine, k: int, original_image: Images) -> list[Images]:
    with Session(engine) as session:
        # execution_options={"prebuffer_rows": True} is used to prebuffer the rows, this is useful when we want to fetch the rows in chunks and return them after session is closed
        result = (
            session.execute(
                select(Images)
                .order_by(
                    Images.image_embedding.cosine_distance(
                        original_image.image_embedding
                    )
                )
                .limit(k),
                execution_options={"prebuffer_rows": True},
            )
            .scalars()
            .all()
        )
        return result


# find the 10 most similar images to the first image
k = 10
similar_images = find_k_images(engine, k, image)

print(f"Original image: {image.image_path}")
for i, img in enumerate(similar_images, start=1):
    print(f"Similar image {i}: {img.image_path}")


Original image: image_0.jpg
Similar image 1: image_0.jpg
Similar image 2: image_46.jpg
Similar image 3: image_61.jpg
Similar image 4: image_72.jpg
Similar image 5: image_16.jpg
Similar image 6: image_65.jpg
Similar image 7: image_59.jpg
Similar image 8: image_33.jpg
Similar image 9: image_34.jpg
Similar image 10: image_63.jpg


In [14]:
# find images with similarity to the original above the threshold (similarity = 1 - cosine_distance)
def find_images_with_similarity_score_greater_than(engine: Engine, similarity_score: float, original_image: Images) -> list[Images]:
    with Session(engine) as session:
        result = session.execute(
            select(Images)
            .filter((1 - Images.image_embedding.cosine_distance(original_image.image_embedding)) > similarity_score),
            execution_options={"prebuffer_rows": True}
        ).scalars().all()
        return result


threshold = 0.77
images_with_high_similarity = find_images_with_similarity_score_greater_than(engine, threshold, image)

print(f"Images with similarity above {threshold}:")
for i, img in enumerate(images_with_high_similarity, start=1):
    print(f"Image {i}: {img.image_path}")

Images with similarity above 0.77:
Image 1: image_0.jpg
Image 2: image_8.jpg
Image 3: image_16.jpg
Image 4: image_23.jpg
Image 5: image_46.jpg
Image 6: image_59.jpg
Image 7: image_64.jpg
Image 8: image_68.jpg
Image 9: image_87.jpg
Image 10: image_33.jpg
Image 11: image_57.jpg
Image 12: image_61.jpg
Image 13: image_63.jpg
Image 14: image_81.jpg
Image 15: image_92.jpg
Image 16: image_34.jpg
Image 17: image_35.jpg
Image 18: image_65.jpg
Image 19: image_72.jpg
Image 20: image_97.jpg


In [15]:
from datasets import load_dataset

dataset = load_dataset("FronkonGames/steam-games-dataset")

# get columns names and types
columns = dataset["train"].features
print(columns)

columns_to_keep = ["name", "windows", "linux", "mac", "detailed_description", "supported_languages", "price"]

N = 40000 # you can adjust this number
dataset = dataset["train"].select_columns(columns_to_keep).select(range(N))


README.md:   0%|          | 0.00/596 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/192M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/124146 [00:00<?, ? examples/s]

{'appID': Value('string'), 'name': Value('string'), 'release_date': Value('string'), 'estimated_owners': Value('string'), 'peak_ccu': Value('int64'), 'required_age': Value('int64'), 'price': Value('float64'), 'dlc_count': Value('int64'), 'detailed_description': Value('string'), 'short_description': Value('string'), 'supported_languages': List(Value('string')), 'full_audio_languages': List(Value('string')), 'reviews': Value('string'), 'header_image': Value('string'), 'website': Value('string'), 'support_url': Value('string'), 'support_email': Value('string'), 'windows': Value('bool'), 'mac': Value('bool'), 'linux': Value('bool'), 'metacritic_score': Value('int64'), 'metacritic_url': Value('string'), 'user_score': Value('int64'), 'positive': Value('int64'), 'negative': Value('int64'), 'score_rank': Value('string'), 'achievements': Value('int64'), 'recommendations': Value('int64'), 'notes': Value('string'), 'average_playtime_forever': Value('int64'), 'average_playtime_2weeks': Value('int6

In [16]:
from sqlalchemy import Boolean, Float, Integer, Text


class Games(Base):
    __tablename__ = "games"
    __table_args__ = {'extend_existing': True}
    
    # the vector size produced by the model taken from documentation https://huggingface.co/sentence-transformers/distiluse-base-multilingual-cased-v2
    VECTOR_LENGTH = 512
        
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    name: Mapped[str] = mapped_column(String(256))
    # `Text` is an unbounded string in Postgres — game descriptions can run a few KB
    description: Mapped[str] = mapped_column(Text)
    windows: Mapped[bool] = mapped_column(Boolean)
    linux: Mapped[bool] = mapped_column(Boolean)
    mac: Mapped[bool] = mapped_column(Boolean)
    price: Mapped[float] = mapped_column(Float)
    game_description_embedding: Mapped[List[float]] = mapped_column(Vector(VECTOR_LENGTH))

Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)


In [17]:
from sentence_transformers import SentenceTransformer

checkpoint = "distiluse-base-multilingual-cased-v2"
model = SentenceTransformer(checkpoint)


def generate_embeddings(text: str) -> list[float]:
    return model.encode(text)


modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/610 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/539M [00:00<?, ?B/s]

... (3 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

In [18]:
from tqdm import tqdm


def insert_games(engine, dataset):
    with tqdm(total=len(dataset)) as pbar:
        for i, game in enumerate(dataset):
            game_description = game["detailed_description"] or ""
            game_embedding = generate_embeddings(game_description)
            name, windows, linux, mac, price = (
                game["name"],
                game["windows"],
                game["linux"],
                game["mac"],
                game["price"],
            )
            # keep rows with the required fields; windows/linux/mac are platform flags (True/False),
            # not missing-data signals — don't gate the insert on them or you discard most of the dataset
            if name and game_description and price is not None:
                game = Games(
                    name=game["name"],
                    description=game_description,
                    windows=game["windows"],
                    linux=game["linux"],
                    mac=game["mac"],
                    price=game["price"],
                    game_description_embedding=game_embedding,
                )
                with Session(engine) as session:
                    session.add(game)
                    session.commit()
            pbar.update(1)


insert_games(engine, dataset)

100%|██████████| 40000/40000 [13:07<00:00, 50.76it/s]


In [21]:
def find_game(
    engine: Engine,
    game_description: str,
    windows: bool | None = None,
    linux: bool | None = None,
    mac: bool | None = None,
    price: int | None = None,
):
    with Session(engine) as session:
        game_embedding = generate_embeddings(game_description)

        query = select(Games).order_by(
            Games.game_description_embedding.cosine_distance(game_embedding)
        )

        if price:
            query = query.filter(Games.price <= price)
        if windows:
            query = query.filter(Games.windows == True)
        if linux:
            query = query.filter(Games.linux == True)
        if mac:
            query = query.filter(Games.mac == True)

        result = session.execute(query, execution_options={"prebuffer_rows": True})
        game = result.scalars().first()

        return game


In [22]:
game = find_game(engine, "This is a game about a hero who saves the world", price=10)
print(f"Game: {game.name}")
print(f"Description: {game.description}")

game = find_game(engine, game_description="Home decorating", price=20)
print(f"Game: {game.name}")
print(f"Description: {game.description}")


Game: Hero's everyday life
Description: In Hero's everyday life, you need to play the role of a hero, completing a variety of tasks from give and bring, to kill monsters and bosses. Help all the different game characters with their mundane tasks and become famous throughout the fantasy kingdom as a great hero! Or just fool around and have as much fun as you can! The world is filled with various secrets, there are also enemies who are ready to break you no matter what, so arm yourself with a sword and hit the road. Who is a hero? Is this the one who gets up in the morning for an adventure? Or the one who fights for the good and prosperity? In general, you will need to decide which of you will be the hero and for what you will fight! The kingdom of fantasy is filled with monsters, NPS cannot fulfill their mission and therefore they called you a hero, and whether you are a great hero or not, it's up to you to decide! Complete a mission, fight monsters, kill bosses, or just insult the king

In [23]:
game = find_game(engine, game_description="Home decorating", mac=True, price=5)
print(f"Game: {game.name}")
print(f"Description: {game.description}")


Game: My Universe - Interior Designer
Description: Paint colours, furnishings, style, interior design, you choose each and every detail. Talk to your clients to find out more about the hopes, dreams and preferences they have for their homes. You're going to have to renovate their houses and complete the mission objectives to satisfy their needs. The happier they are with your work, the more your notoriety and level will increase! Gradually, you'll unlock new items, styles, decorative possibilities and bigger projects. During your adventure, you can play in 2 different modes: renovation mode where you can change the structure of the house by knocking down or putting up walls, taking up wooden or tiled flooring, painting, changing the wall coverings, fixing the electrics or adding new water inlets. You can also switch to decoration mode where you can choose and position the furnishings, plants, rugs, wallpaper, curtains, etc. It's up to you to add real character to the house. And if you 

## Retrieval-Augmented Generation (RAG) service

In [24]:
from pymilvus import MilvusClient

milvus_client = MilvusClient("./milvus_db/milvus_rag.db")


In [25]:
from pymilvus import CollectionSchema, DataType, FieldSchema

VECTOR_LENGTH = 768  # check the dimensionality for Silver Retriever Base (v1.1) model

id_field = FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, description="Primary id")
text = FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096, description="Page text")
embedding_text = FieldSchema("embedding", dtype=DataType.FLOAT_VECTOR, dim=VECTOR_LENGTH, description="Embedded text")

fields = [id_field, text, embedding_text]

schema = CollectionSchema(fields=fields, auto_id=True, enable_dynamic_field=True, description="RAG Texts collection")


In [26]:
COLLECTION_NAME = "rag_texts_and_embeddings"

milvus_client.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema
)

index_params = milvus_client.prepare_index_params()

index_params.add_index(
    field_name="embedding", 
    index_type="HNSW",
    metric_type="L2",
    params={"M": 4, "efConstruction": 64}  # lower values for speed
) 

milvus_client.create_index(
    collection_name=COLLECTION_NAME,
    index_params=index_params
)

# checkout our collection
print(milvus_client.list_collections())

# describe our collection
print(milvus_client.describe_collection(COLLECTION_NAME))


['rag_texts_and_embeddings']
{'collection_name': 'rag_texts_and_embeddings', 'auto_id': True, 'num_shards': 1, 'description': '', 'fields': [{'field_id': 0, 'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 0, 'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 4096}}, {'field_id': 0, 'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'functions': [], 'aliases': [], 'collection_id': 0, 'consistency_level': 0, 'properties': {}, 'num_partitions': 1, 'enable_dynamic_field': True}


In [27]:
# define data source and destination
## the document origin destination from which document will be downloaded 
pdf_url = "https://www.iab.org.pl/wp-content/uploads/2024/04/Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the document
file_name = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the processed document 
file_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.json"

## local destination of the embedded pages of the document
embeddings_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska-Embeddings.json"

## local destination of all above local required files
data_dir = "./data"


In [28]:
# download data
import os

import requests


def download_pdf_data(pdf_url: str, file_name: str) -> None:
    response = requests.get(pdf_url, stream=True)
    os.makedirs(data_dir, exist_ok=True)
    with open(os.path.join(data_dir, file_name), "wb") as file:
        for block in response.iter_content(chunk_size=1024):
            if block:
                file.write(block)

download_pdf_data(pdf_url, file_name)


In [29]:
# prepare data

import json

import fitz


def extract_pdf_text(file_name, file_json):
    document = fitz.open(os.path.join(data_dir, file_name))
    pages = []

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        page_text = page.get_text()
        pages.append({"page_num": page_num, "text": page_text})

    with open(os.path.join(data_dir, file_json), "w") as file:
        json.dump(pages, file, indent=4, ensure_ascii=False)


extract_pdf_text(file_name, file_json)


In [30]:
# vectorize data

import numpy as np
import torch
from sentence_transformers import SentenceTransformer


def generate_embeddings(file_json, embeddings_json, model):
    pages = []
    with open(os.path.join(data_dir, file_json), "r") as file:
        data = json.load(file)

    for page in data:
        pages.append(page["text"])

    embeddings = model.encode(pages)

    embeddings_paginated = []
    for page_num in range(len(embeddings)):
        embeddings_paginated.append({"page_num": page_num, "embedding": embeddings[page_num].tolist()})

    with open(os.path.join(data_dir, embeddings_json), "w") as file:
        json.dump(embeddings_paginated, file, indent=4, ensure_ascii=False)

model_name = "ipipan/silver-retriever-base-v1.1"
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)
generate_embeddings(file_json, embeddings_json, model)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

... (2 outputs left out)

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/144 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [31]:
def insert_embeddings(file_json, embeddings_json, client=milvus_client):
    rows = []
    with open(os.path.join(data_dir, file_json), "r") as t_f, open(os.path.join(data_dir, embeddings_json), "r") as e_f:
        text_data, embedding_data = json.load(t_f), json.load(e_f)
        text_data =  list(map(lambda d: d["text"], text_data))
        embedding_data = list(map(lambda d: d["embedding"], embedding_data))
        
        for page, (text, embedding) in enumerate(zip(text_data, embedding_data)):
            rows.append({"text":text, "embedding": embedding})

    client.insert(collection_name="rag_texts_and_embeddings", data=rows)


insert_embeddings(file_json, embeddings_json)

# load inserted data into memory
milvus_client.load_collection("rag_texts_and_embeddings")


In [37]:
# search

def search(model, query, client=milvus_client):
    embedded_query = model.encode(query).tolist()
    result = client.search(
        collection_name="rag_texts_and_embeddings", 
        data=[embedded_query], 
        limit=1,
        search_params={"metric_type": "L2"},
        output_fields=["text"]
    )
    return result


result = search(model, query="Czym jest sztuczna inteligencja")
print("Query: Czym jest sztuczna inteligencja")
print(f"Result: {result[0][0]['text'][:100]}...")


Query: Czym jest sztuczna inteligencja
Result: Historia powstania
sztucznej inteligencji
7
W języku potocznym „sztuczny" oznacza to, co
jest 
wytwo...


## Gemini API integration

In [38]:
from dotenv import load_dotenv

load_dotenv()

True

In [39]:
from google import genai

GEMINI_KEY = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=GEMINI_KEY)

MODEL = "gemini-2.5-flash"

def generate_response(prompt: str):
    try:
        # Send request to Gemini 2.5 Flash API and get the response
        response = gemini_client.models.generate_content(
            model=MODEL,
            contents=prompt,
        )
        return response.text 
    except Exception as e:
        print(f"Error generating response: {e}")
        return None

print(generate_response("Test"))

Received!

How can I help you today? What would you like to test or what are you interested in discussing?


In [57]:
def build_prompt(context: str, query: str) -> str:
    prompt = f"""
    You are a helpful assistant. Use the following pieces of context to answer the question at the end.
    If you don't know the answer, just say that you don't know, don't try to make up an answer.

    <context>
    {context}
    </context>

    <query>
    {query}
    </query>
    """
    return prompt
    

def rag(model, query: str) -> str:
    # having all prepared functions, you can combine them together and try to build your own RAG!
    result = search(model, query="Czym jest sztuczna inteligencja")
    if result:
        context = result[0][0]['text']
        source_id = result[0][0].get('id', None)

        prompt = build_prompt(context, query)
        return generate_response(prompt) + f"\nSource ID: {source_id}"
    return "No relevant information found."

### Testing

In [58]:
print(rag(model, "Czym jest sztuczna inteligencja?"))

Sztuczna inteligencja (AI) to obszar informatyki, który skupia się na tworzeniu programów komputerowych zdolnych do wykonywania zadań wymagających ludzkiej inteligencji. Jest to program lub maszyna symulująca naturalną, ludzką inteligencję.

Zadania te obejmują:
*   rozpoznawanie wzorców
*   rozumienie języka naturalnego
*   podejmowanie decyzji
*   uczenie się
*   planowanie

Głównym celem AI jest stworzenie systemów zdolnych do myślenia i podejmowania decyzji w sposób przypominający ludzki, a czasem nawet przewyższający funkcje naturalne (np. będąc wolnym od pomyłek przy liczeniu czy defektów pamięci).
Source ID: 7


### TASK: Experiment with the prompt - small wording changes can dramatically reshape the output. Try at least 2–3 of:

* Style: force pirate speak, Shakespearean English, Yoda, or corporate jargon.
* Language: instruct the model to answer in Polish (or Japanese, Latin, …) even though the retrieved context is in English.
* Format: force a haiku, a single sentence, exactly three bullet points, or "emoji + one-line explanation" per fact.
* Persona: a grumpy professor; an over-enthusiastic YouTuber; a skeptical reviewer that grades each claim "supported / unsupported by context".
* Constraints: answer without using the letter "e"; max 50 words; each sentence starts with the next letter of the alphabet.
* Hallucination probe: ask something not in the retrieved context and check whether the model invents an answer or honestly says "not in sources". A good RAG prompt should make refusal easy.


Note which prompt tweaks survive retrieval (model stays grounded) and which make it drift away from the context. That tension is the core RAG craft.

In [59]:
query = "Czym jest sztuczna inteligencja?"
result = search(model, query)
context = result[0][0]["text"]
print("Kontekst:", context)

experiments = [
    ("Styl", query, "Odpowiedz po polsku w stylu pirata."),
    ("Format", query, "Odpowiedz po polsku w formie Trzynastozgłoskowca"),
    (
        "Persona",
        query,
        """
        Jesteś bardzo starym i bardzo zmęczonym profesorem informatyki, który ma problemy z pamięcią i często zapomina o czym mówi.
        Odpowiedz po polsku w tym stylu.
        """,
    ),
    (
        "Pytanie spoza źródła",
        "Co autor tego dokumentu zjadł dziś na śniadanie?",
        'Odpowiedz po polsku. Jeśli kontekst nie zawiera odpowiedzi, napisz tylko "Brak informacji w źródle".',
    ),
]

def build_experiment_prompt(instruction: str, context: str, query: str) -> str:
    prompt = f"""
    {instruction}

    <context>
    {context}
    </context>

    <query>
    {query}
    </query>
    """
    return prompt

responses = []
for i, (name, question, instruction) in enumerate(experiments):
    prompt = build_experiment_prompt(instruction, context, question)
    response = generate_response(prompt)
    if response is None:
        break
    print(f"==== Experiment {i + 1} ====")
    print(f"\n{name}:\n{response}")
    responses.append(f"{name}: {response}")

Kontekst: Historia powstania
sztucznej inteligencji
7
W języku potocznym „sztuczny" oznacza to, co
jest 
wytworem 
mającym 
naśladować 
coś
naturalnego. W takim znaczeniu używamy
terminu ,,sztuczny'', gdy mówimy o sztucznym
lodowisku lub oku. Sztuczna inteligencja byłaby
czymś (programem, maszyną) symulującym
inteligencję naturalną, ludzką.
Sztuczna inteligencja (AI) to obszar informatyki,
który skupia się na tworzeniu programów
komputerowych zdolnych do wykonywania
zadań, które wymagają ludzkiej inteligencji. 
Te zadania obejmują rozpoznawanie wzorców,
rozumienie języka naturalnego, podejmowanie
decyzji, uczenie się, planowanie i wiele innych.
Głównym celem AI jest stworzenie systemów,
które są zdolne do myślenia i podejmowania
decyzji na sposób przypominający ludzki.
Historia sztucznej inteligencji sięga lat 50. 
XX wieku, kiedy to powstały pierwsze koncepcje
i modele tego, co mogłoby stać się sztuczną
inteligencją. Jednym z pionierów był Alan
Turing, który sformułował test Turinga, 

### Note which prompt tweaks survive retrieval (model stays grounded) and which make it drift away from the context.

- Styl pirata zachowuje główne fakty ze źródła. Porównanie do czytania w myślach jest przesadzone.
- Wiersz zachowuje ogólny sens definicji AI, ale pomija szczegóły. Nie dodaje wyraźnych halucynacji.
- Profesor podaje fakty ze źródła, ale miesza daty. Dodaje też zmyślone wspomnienia swojej postaci.
- Przy pytaniu o śniadanie model odpowiada „Brak informacji w źródle”. Ten wariant pozostaje grounded i nie halucynuje.
